# DeepSentinel — Colab Training (Phase 1 + Phase 2)

## Files to upload to Google Drive before running

### Phase 1 — 172 MB total (frozen backbones, cached features only)
| Zip | Contents | Size |
|---|---|---|
| `existing_features.zip` | `data/preprocessed/features/` (z_at + z_v, 14k clips) | ~172 MB |
| `metadata.zip` | 4 track CSVs + meld_real.csv + mosei_real.csv + sarcasm_data.json | ~3.3 MB |
| `mosei_features.zip` | MOSEI z_at + z_v from `colab_mosei_preprocess.ipynb` | ~25 MB |

### Phase 2 — additional ~8 GB (backbone fine-tune, needs raw audio + video)
| Zip | Contents | Size |
|---|---|---|
| `audio_cache.zip` | `data/preprocessed/audio/` (WAVs) | ~1.5 GB |
| `transcripts.zip` | `data/preprocessed/transcripts/` (.txt) | ~0.7 MB |
| `track1_clips.zip` | `data/synthetic/track1_fakes/` | ~595 MB |
| `track2_clips.zip` | `data/synthetic/track2_fakes/` | ~868 MB |
| `track3_clips.zip` | `data/synthetic/track3_fakes/` | ~1.3 GB |
| `meld_real_clips.zip` | MELD real clips only (those in meld_real.csv) | ~3 GB |
| `mustard_clips.zip` | `data/raw/MUStARD/raw_data/utterances_final/` | ~382 MB |

> Track 4 is excluded (`--no_track4`) — do NOT upload track4 clips.

---
## Local PowerShell commands to create all zips
```powershell
# Phase 1
Compress-Archive -Path data/preprocessed/features -DestinationPath existing_features.zip

# metadata.zip — must be created with Python (Compress-Archive can't easily multi-source)
python -c "
import zipfile, shutil
from pathlib import Path
files = [
    ('data/synthetic/track1_fakes/metadata.csv', 'data/synthetic/track1_fakes/metadata.csv'),
    ('data/synthetic/track2_fakes/metadata.csv', 'data/synthetic/track2_fakes/metadata.csv'),
    ('data/synthetic/track3_fakes/metadata.csv', 'data/synthetic/track3_fakes/metadata.csv'),
    ('data/synthetic/track4_fakes/metadata.csv', 'data/synthetic/track4_fakes/metadata.csv'),
    ('data/processed/meld_manifests/meld_real.csv', 'data/processed/meld_manifests/meld_real.csv'),
    ('data/processed/mosei_manifests/mosei_real.csv', 'data/processed/mosei_manifests/mosei_real.csv'),
    ('data/raw/MUStARD/repo/data/sarcasm_data.json', 'data/raw/MUStARD/repo/data/sarcasm_data.json'),
]
with zipfile.ZipFile('metadata.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for src, arc in files:
        p = Path(src)
        if p.exists(): zf.write(p, arc)
print('metadata.zip done')
"

# Phase 2 zips
Compress-Archive -Path data/preprocessed/audio        -DestinationPath audio_cache.zip
Compress-Archive -Path data/preprocessed/transcripts  -DestinationPath transcripts.zip
Compress-Archive -Path data/synthetic/track1_fakes    -DestinationPath track1_clips.zip
Compress-Archive -Path data/synthetic/track2_fakes    -DestinationPath track2_clips.zip
Compress-Archive -Path data/synthetic/track3_fakes    -DestinationPath track3_clips.zip
Compress-Archive -Path data/raw/MUStARD/raw_data/utterances_final -DestinationPath mustard_clips.zip

# MELD real clips only (saves space vs zipping all of MELD)
python -c "
import pandas as pd, zipfile
from pathlib import Path
df = pd.read_csv('data/processed/meld_manifests/meld_real.csv')
with zipfile.ZipFile('meld_real_clips.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for vp in df['video_path']:
        p = Path(vp)
        if p.exists(): zf.write(p, str(p.relative_to('.')))
print('meld_real_clips.zip done')
"
```

In [ ]:
# ── CONFIG — edit before running ───────────────────────────────────────────────
DRIVE_DIR        = "/content/drive/MyDrive/deepsentinel_train"       # folder with all zips
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/deepsentinel_checkpoints" # output for checkpoints

REPO_URL    = "https://github.com/gjvlio/emotion-based-multimodal-deepfake-detector.git"
REPO_BRANCH = "exp/sarcasm-training"

# Local Windows repo root — used to remap video paths for Phase 2
LOCAL_REPO_ROOT = r"D:\Documents\Programming\Thesis_G10"
# ──────────────────────────────────────────────────────────────────────────────

## Step 1 — Mount Drive + install deps

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"Drive mounted.")
print(f"  Input  : {DRIVE_DIR}")
print(f"  Output : {DRIVE_OUTPUT_DIR}")

In [ ]:
!pip install -q transformers scikit-learn tensorboard timm
print("Deps installed.")

## Step 2 — Clone repo

In [ ]:
import subprocess, os
REPO_DIR = "/content/thesis"

if os.path.exists(REPO_DIR):
    print("Repo exists — pulling...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print(f"Cloning {REPO_URL} @ {REPO_BRANCH} ...")
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )
os.chdir(REPO_DIR)
import sys
sys.path.insert(0, REPO_DIR)
print(f"CWD: {os.getcwd()}")

## Step 3 — Extract all data

In [ ]:
import zipfile, os
from pathlib import Path

REPO_DIR = "/content/thesis"
os.chdir(REPO_DIR)

def extract_zip(zip_name, extract_to=REPO_DIR, optional=False):
    zip_path = os.path.join(DRIVE_DIR, zip_name)
    if not os.path.exists(zip_path):
        tag = "OPTIONAL — SKIP" if optional else "MISSING — REQUIRED"
        print(f"  [{tag}] {zip_name}")
        return False
    size_mb = os.path.getsize(zip_path) / 1e6
    print(f"  Extracting {zip_name} ({size_mb:.0f} MB) ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_to)
    print(f"  Done.")
    return True

# Create all expected dirs
for d in [
    "data/preprocessed/features/z_at",
    "data/preprocessed/features/z_v",
    "data/preprocessed/audio",
    "data/preprocessed/transcripts",
    "data/preprocessed/keyframes",
    "data/processed/meld_manifests",
    "data/processed/mosei_manifests",
    "data/synthetic/track1_fakes",
    "data/synthetic/track2_fakes",
    "data/synthetic/track3_fakes",
    "data/raw/MUStARD/repo/data",
    "checkpoints/full",
    "logs/full",
]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("=== Phase 1 data ===")
extract_zip("existing_features.zip")
extract_zip("metadata.zip")
extract_zip("mosei_features.zip", extract_to="data/preprocessed")

print()
print("=== Phase 2 data (skipped if not uploaded) ===")
extract_zip("audio_cache.zip",    optional=True)
extract_zip("transcripts.zip",    optional=True)
extract_zip("track1_clips.zip",   optional=True)
extract_zip("track2_clips.zip",   optional=True)
extract_zip("track3_clips.zip",   optional=True)
extract_zip("meld_real_clips.zip",optional=True)
extract_zip("mustard_clips.zip",  optional=True)

## Step 4 — Verify features

In [ ]:
import torch
from pathlib import Path

z_at = list(Path("data/preprocessed/features/z_at").glob("*.pt"))
z_v  = list(Path("data/preprocessed/features/z_v").glob("*.pt"))
wavs = list(Path("data/preprocessed/audio").glob("*.wav"))

print(f"z_at .pt files : {len(z_at):>6}  (expect 14,000+)")
print(f"z_v  .pt files : {len(z_v):>6}  (expect 14,000+)")
print(f"audio WAVs     : {len(wavs):>6}  (0 = Phase 2 audio not uploaded yet)")

if not z_at:
    raise RuntimeError("No z_at features — existing_features.zip not extracted correctly")

s_at = torch.load(z_at[0], weights_only=True)
s_v  = torch.load(z_v[0],  weights_only=True)
print(f"\nz_at shape: {s_at.shape}  (expect [1536])")
print(f"z_v  shape: {s_v.shape}   (expect [768])")
print(f"GPU: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

# Check metadata CSVs
for c in [
    "data/synthetic/track1_fakes/metadata.csv",
    "data/synthetic/track2_fakes/metadata.csv",
    "data/synthetic/track3_fakes/metadata.csv",
    "data/processed/meld_manifests/meld_real.csv",
    "data/processed/mosei_manifests/mosei_real.csv",
]:
    p = Path(c)
    n = sum(1 for _ in open(p)) - 1 if p.exists() else -1
    print(f"  {p.name:<30} {'OK ' + str(n) + ' rows' if n >= 0 else 'MISSING'}")

---
# Phase 1 — Frozen Backbones

Trains detection head + emotion heads + sarcasm head using cached z_at/z_v features.  
Does NOT touch backbone weights. Fast (~15–30 min on T4).  
**Only requires `existing_features.zip` + `metadata.zip` + `mosei_features.zip`.**

In [ ]:
import os
os.chdir("/content/thesis")

!python scripts/train_full.py \
    --device cuda \
    --no_track4 \
    --no_phase2 \
    --batch_size 32 \
    --epochs 50 \
    --patience 7 \
    --lr 1e-3 \
    2>&1

In [ ]:
import shutil, os
from pathlib import Path

ckpt = Path("checkpoints/full/best_phase1.pt")
if not ckpt.exists():
    raise FileNotFoundError(f"Checkpoint not found: {ckpt} — check training output above")

dst = os.path.join(DRIVE_OUTPUT_DIR, "best_phase1.pt")
shutil.copy2(str(ckpt), dst)
print(f"Phase 1 checkpoint saved → {dst}  ({ckpt.stat().st_size/1e6:.1f} MB)")

logs = Path("logs/full")
if logs.exists() and any(logs.iterdir()):
    shutil.copytree(str(logs), os.path.join(DRIVE_OUTPUT_DIR, "logs_phase1"), dirs_exist_ok=True)
    print("Logs saved to Drive.")

---
# Phase 2 — Backbone Fine-Tuning

Unfreezes top 2 transformer layers per backbone (Wav2Vec2, BERT, ViT) and fine-tunes end-to-end.  
**Requires the Phase 2 zips to be uploaded** (audio_cache, transcripts, track1-3 clips, MELD, MUStARD).  

> **Alternative if Phase 2 zips are not uploaded:**  
> Download `best_phase1.pt` from Drive and run Phase 2 locally:
> ```powershell
> python scripts/train_full.py --device cuda --no_track4 --patience 5 `
>     --phase2_epochs 3 --phase2_batch 1 --phase2_freeze_layers 2 --no_grad_ckpt
> ```
> (Configured for 6 GB VRAM — your RTX 4050 can handle this.)

In [ ]:
# ── Phase 2 prerequisite check ─────────────────────────────────────────────────
import os
from pathlib import Path

required = [
    ("audio_cache.zip",     "~1.5 GB"),
    ("transcripts.zip",     "~0.7 MB"),
    ("track1_clips.zip",    "~595 MB"),
    ("track2_clips.zip",    "~868 MB"),
    ("track3_clips.zip",    "~1.3 GB"),
    ("meld_real_clips.zip", "~3 GB"),
    ("mustard_clips.zip",   "~382 MB"),
]

all_ok = True
for fname, expected_size in required:
    p = os.path.join(DRIVE_DIR, fname)
    if os.path.exists(p):
        mb = os.path.getsize(p) / 1e6
        print(f"  OK     {fname:<28} {mb:>7.0f} MB")
    else:
        print(f"  MISS   {fname:<28} (expected {expected_size})")
        all_ok = False

if all_ok:
    print("\nAll Phase 2 files present. Run next cells.")
else:
    print("\nMissing files — upload to Drive first, or run Phase 2 locally (see note above).")

In [ ]:
# ── Remap Windows video paths → Colab paths in all metadata CSVs ───────────────
import pandas as pd
from pathlib import Path

COLAB_ROOT = "/content/thesis"
LOCAL_ROOT = LOCAL_REPO_ROOT.replace("\\", "/")   # D:/Documents/Programming/Thesis_G10

def remap_csv(csv_path, path_cols):
    p = Path(csv_path)
    if not p.exists():
        print(f"  SKIP (not found): {csv_path}")
        return
    df = pd.read_csv(p)
    changed = 0
    for col in path_cols:
        if col not in df.columns:
            continue
        def fix(v):
            if not isinstance(v, str): return v
            v = v.replace("\\", "/")
            return v.replace(LOCAL_ROOT, COLAB_ROOT) if LOCAL_ROOT in v else v
        before = df[col].copy()
        df[col] = df[col].map(fix)
        changed += (df[col] != before).sum()
    df.to_csv(p, index=False)
    print(f"  {p.name:<35} {changed} paths remapped")

print("Remapping video paths...")
remap_csv("data/synthetic/track1_fakes/metadata.csv", ["output_path", "input_path"])
remap_csv("data/synthetic/track2_fakes/metadata.csv", ["output_path", "input_path"])
remap_csv("data/synthetic/track3_fakes/metadata.csv", ["output_path", "input_path"])
remap_csv("data/processed/meld_manifests/meld_real.csv",  ["video_path"])
remap_csv("data/processed/mosei_manifests/mosei_real.csv", ["video_path"])
print("Done.")

In [ ]:
# ── Spot-check remapped path ────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

df = pd.read_csv("data/synthetic/track1_fakes/metadata.csv")
sample = df["output_path"].iloc[0]
print(f"Sample remapped path: {sample}")
ok = Path(sample).exists()
print(f"File exists: {ok}")

if not ok:
    print("\nFix: Check that:")
    print(f"  1. track1_clips.zip was extracted (Step 3)")
    print(f"  2. LOCAL_REPO_ROOT in config cell matches your Windows repo root")
    print(f"     Current value: {LOCAL_REPO_ROOT}")

In [ ]:
# ── Phase 2 Training — Python API (loads Phase 1 checkpoint, skips re-running Phase 1) ──
import os, sys, torch
from pathlib import Path
from torch.utils.data import DataLoader

os.chdir("/content/thesis")
sys.path.insert(0, "/content/thesis")

from src.models.detection_model import DeepfakeDetector
from src.training.trainer import Trainer
from src.utils.config import Config
from scripts.train_full import build_datasets

DEVICE = "cuda"
PREPROCESSED_DIR   = Path("data/preprocessed")
KEYFRAME_CACHE_DIR = Path("data/preprocessed/keyframes")
CKPT_DIR           = Path("checkpoints/full")
LOG_DIR            = Path("logs/full")

cfg = Config.from_yaml()

# Build datasets (speaker-stratified split)
train_ds, val_ds, test_ds, stats, Phase2FullDataset, p2_train_recs = build_datasets(
    cfg, seed=42, include_track4=False
)
print(f"Train: {stats['train']}  Val: {stats['val']}  Test: {stats['test']}")
print(f"Real: {stats['real_train']}  Fake: {stats['fake_train']}  Sarc: {stats['sarc_train']}")

# Phase 2 DataLoader — raw audio/text/frames (NOT cached features)
# T4 has 15 GB VRAM; batch 2 is safe (local PC uses batch 1 for 6 GB)
p2_ds     = Phase2FullDataset(p2_train_recs, PREPROCESSED_DIR, KEYFRAME_CACHE_DIR)
p2_loader = DataLoader(p2_ds, shuffle=True, batch_size=2, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

# Load model with Phase 1 weights
model    = DeepfakeDetector().to(DEVICE)
ckpt_src = CKPT_DIR / "best_phase1.pt"
if not ckpt_src.exists():
    # Try loading from Drive if not already local
    import shutil
    drive_ckpt = os.path.join(DRIVE_OUTPUT_DIR, "best_phase1.pt")
    if os.path.exists(drive_ckpt):
        shutil.copy2(drive_ckpt, str(ckpt_src))
        print(f"Copied Phase 1 checkpoint from Drive.")
    else:
        raise FileNotFoundError(
            "best_phase1.pt not found. Run Phase 1 training cells above first."
        )

# Checkpoint is a dict: {"val_loss": ..., "epoch": ..., "model_state": ...}
ckpt_data = torch.load(str(ckpt_src), map_location=DEVICE, weights_only=True)
model.load_state_dict(ckpt_data["model_state"], strict=False)
print(f"Loaded Phase 1 checkpoint (epoch={ckpt_data['epoch']}, val_loss={ckpt_data['val_loss']:.4f})")

# Trainer — p2_loader as train_loader (Trainer.__init__ calls len(train_loader))
trainer = Trainer(
    model          = model,
    train_loader   = p2_loader,
    val_loader     = val_loader,
    checkpoint_dir = CKPT_DIR,
    log_dir        = LOG_DIR,
    fp16           = True,
    lambda_a       = 0.5,
    lambda_b       = 0.5,
    lambda_sarcasm = 0.3,
    pos_weight     = stats["auto_pos_weight"],
    device         = DEVICE,
)

# Phase 2: unfreeze top 2 transformer layers, grad checkpointing ON
# T4 = 15 GB VRAM, so grad_ckpt=True is safe and saves memory
trainer.train_phase2(
    lr            = 1e-5,
    weight_decay  = 1e-4,
    max_epochs    = 5,
    patience      = 5,
    freeze_layers = 2,
    grad_ckpt     = True,
)
print("Phase 2 complete.")

In [ ]:
import shutil, os
from pathlib import Path

for fname in ["best_phase2.pt", "best_phase1.pt"]:
    src = Path(f"checkpoints/full/{fname}")
    if src.exists():
        dst = os.path.join(DRIVE_OUTPUT_DIR, fname)
        shutil.copy2(str(src), dst)
        print(f"Saved {fname} → Drive  ({src.stat().st_size/1e6:.1f} MB)")

# Keyframe cache (only save if under 5 GB — will be large after Phase 2 runs)
kf_dir   = Path("data/preprocessed/keyframes")
kf_files = list(kf_dir.glob("*.pt"))
if kf_files:
    kf_mb = sum(f.stat().st_size for f in kf_files) / 1e6
    print(f"\nKeyframe cache: {len(kf_files)} files, {kf_mb:.0f} MB")
    if kf_mb < 5000:
        shutil.make_archive("/content/keyframe_cache", "zip", kf_dir.parent, kf_dir.name)
        shutil.copy2("/content/keyframe_cache.zip", os.path.join(DRIVE_OUTPUT_DIR, "keyframe_cache.zip"))
        print("Keyframe cache saved to Drive.")
    else:
        print("Keyframe cache > 5 GB — too large to save to Drive automatically.")

# Logs
logs = Path("logs/full")
if logs.exists():
    shutil.copytree(str(logs), os.path.join(DRIVE_OUTPUT_DIR, "logs_phase2"), dirs_exist_ok=True)
    print("Logs saved to Drive.")

---
## After training — local steps

Download from Drive and place in repo:
```
best_phase1.pt  →  checkpoints/full/best_phase1.pt
best_phase2.pt  →  checkpoints/full/best_phase2.pt
```

Evaluate on FakeAVCeleb:
```powershell
python scripts/evaluate_fakeavceleb.py --checkpoint checkpoints/full/best_phase2.pt
```